# 07 Model Selection

Notebooks 01 to 06 produced dozens of forecasts for the same products and weeks. This notebook asks how to use them:

- Should every product use the same model, or should each product, department or class get its own?
- Does choosing per product, or averaging several models, beat simply using the best single model?

All choices are made from backtests that were complete on the decision date, and nothing is retrained. We work in two business scenarios:

- **Without future drivers**: the price and promotion plan is not available (for example, a manufacturer forecasting a retailer's sales).
- **With future drivers**: the plan is available for the 14 forecast weeks (the retailer forecasting its own sales).

Sections 1 to 5 are the live session and run in about three to four minutes. Optional sections A to F test the conclusions further.

In [1]:
import os
import sys
import warnings

if "google.colab" in sys.modules:
    !git clone -q https://github.com/tackes/time-series-forecasting-bootcamp
    %cd time-series-forecasting-bootcamp
    %pip install -q statsforecast==2.1.1 mlforecast==1.1.0 utilsforecast==0.2.17 lightgbm==4.7.0 catboost==1.2.10
elif not os.path.exists("shared"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore")

In [2]:
import numpy as np
import pandas as pd
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mae

from shared.prepare_data import load_panel, DEV_CUTOFFS, TEST_CUTOFF, STABILITY_CUTOFFS, completed_runs
from shared.evaluation import common_rows, to_wide
from shared.leaderboard import load_forecasts, save_forecasts, leaderboard
from shared.model_selection import (LEVELS, usable_models, hierarchy, selection_evidence, choose, strategy, average, top3,
                                    best_individual, wape_by_cutoff, ensemble_evidence, residuals, ensemble_rules, mean_correlation)

panel = load_panel()
actuals = panel[["unique_id", "ds", "y"]]

## 1. Candidates

We load the forecasts saved by notebooks 01 to 06 and split them by what they need to know on the forecast date:

- **Statistical baselines**: Naive, HistoricAverage and SeasonalNaive.
- **ML without future drivers**: LightGBM models that use sales history, calendar, item attributes and past prices and promotions.
- **ML with future drivers**: models that also read the price and promotion plan. They are candidates only in the second scenario.

Leakage diagnostics from notebook 03 are never candidates. Weaker models stay in on purpose: a selection method should cope with them.

In [3]:
files = ["01_baselines", "02_ml_baseline", "03_exogenous", "03b_promotion_spikes", "04_global_vs_specialized", "05_horizon_strategies", "06_objectives", "06_l1_features"]
forecasts = load_forecasts(files)
forecasts = forecasts[~forecasts.model.isin(["Selected_per_SKU", "Average"])]    # preliminary strategies from notebook 01

stats = ["Naive", "HistoricAverage", "SeasonalNaive"]
ml = [m for m in usable_models(forecasts, future_drivers=False) if m not in stats]
future = [m for m in usable_models(forecasts, future_drivers=True) if m not in stats + ml]

candidates = forecasts[forecasts.model.isin(stats + ml + future)]
print(f"{len(stats)} statistical, {len(ml)} ML without future drivers, {len(future)} ML with future drivers")
print("not candidates:", sorted(set(forecasts.model) - set(candidates.model)))

3 statistical, 16 ML without future drivers, 8 ML with future drivers


not candidates: []


Development leaderboard of every candidate: WAPE over the seven development runs, grouped by the information each model needs. These are the models selection will choose from.

In [4]:
columns = ["model", "information", "coverage", "wape", "bias", "mean_rmsse", "median_rmsse", "apc"]
dev_board = leaderboard(candidates, actuals, scope="dev")
dev_best = dev_board[dev_board.model.isin(stats + ml)].sort_values("wape").model.iloc[0]
dev_best_future = dev_board.sort_values("wape").model.iloc[0]
dev_board[columns].groupby(dev_board.information_tier, sort=False).head(30).round(3)

,model,information,coverage,wape,bias,mean_rmsse,median_rmsse,apc
0,lgbm_l1_std_categorical,Sales history and calendar only,1.000,0.358,-0.072,1.728,0.830,0.127
1,lgbm_l1_std,Sales history and calendar only,1.000,0.378,-0.039,1.705,0.833,0.143
2,lgbm_direct,Sales history and calendar only,1.000,0.398,-0.033,1.765,0.873,0.124
3,SeasonalNaive,Sales history and calendar only,0.988,0.408,-0.042,1.850,1.055,0.000
4,lgbm_l1_raw,Sales history and calendar only,1.000,0.409,-0.143,1.757,0.858,0.183
5,lgbm_categorical,Sales history and calendar only,1.000,0.423,0.103,1.851,0.891,0.138
6,HistoricAverage,Sales history and calendar only,1.000,0.426,-0.060,1.791,0.887,0.015
7,lgbm_horizon_safe,Sales history and calendar only,1.000,0.427,-0.012,1.806,0.907,0.087
8,lgbm_tweedie_raw,Sales history and calendar only,1.000,0.432,0.008,1.918,0.951,0.197
9,lgbm_dept_std,Sales history and calendar only,1.000,0.440,0.119,1.865,0.918,0.156


## 2. Selection by hand

The simplest tournament: for each SKU, pick the candidate with the lowest MAE over the seven development runs, then use that candidate's forecast at the final cutoff. Candidates are the models without future drivers.

`evaluate` with the SKU as the series id gives one MAE per SKU and candidate. That is the selection error. The leaderboard is a different calculation: it pools errors over all SKUs into one WAPE.

In [5]:
print(sorted(stats + ml))

['HistoricAverage', 'Naive', 'SeasonalNaive', 'lgbm_baseline', 'lgbm_categorical', 'lgbm_dept_raw', 'lgbm_dept_std', 'lgbm_direct', 'lgbm_global_minmax', 'lgbm_hist_drivers', 'lgbm_horizon_safe', 'lgbm_l1_hist_drivers', 'lgbm_l1_raw', 'lgbm_l1_std', 'lgbm_l1_std_categorical', 'lgbm_quantile60_std', 'lgbm_raw', 'lgbm_tweedie_raw', 'lgbm_velocity_std']


In [6]:
evidence_rows = common_rows(candidates[candidates.model.isin(stats + ml) & candidates.cutoff.isin(DEV_CUTOFFS)]).merge(actuals, on=["unique_id", "ds"])
sku_mae = evaluate(to_wide(evidence_rows), metrics=[mae], models=sorted(stats + ml)).drop(columns="metric").set_index("unique_id")
sku_winner = sku_mae.idxmin(axis=1)
sku_mae.iloc[:3, ].round(1)

,HistoricAverage,Naive,SeasonalNaive,lgbm_baseline,lgbm_categorical,lgbm_dept_raw,lgbm_dept_std,lgbm_direct,lgbm_global_minmax,lgbm_hist_drivers,lgbm_horizon_safe,lgbm_l1_hist_drivers,lgbm_l1_raw,lgbm_l1_std,lgbm_l1_std_categorical,lgbm_quantile60_std,lgbm_raw,lgbm_tweedie_raw,lgbm_velocity_std
unique_id,,,,,,,,,,,,,,,,,,,
10006_S1,646.1,1106.3,572.4,747.9,742.2,738.4,746.4,690.8,826.3,750.0,648.1,636.4,674.0,681.7,641.9,834.7,754.1,819.7,719.2
10008_S1,263.8,319.3,356.0,250.2,245.4,228.7,301.8,231.3,367.9,250.3,230.6,286.6,246.3,229.7,287.5,277.4,225.4,245.2,241.7
10010_S1,2272.1,714.3,1870.6,1345.8,1582.1,1984.5,1535.1,1797.0,780.6,1574.9,1943.0,1104.1,711.6,1144.1,1158.1,802.4,753.6,827.2,1221.2


In [7]:
sku_winner.value_counts()

SeasonalNaive              310
HistoricAverage            268
lgbm_l1_raw                211
lgbm_l1_hist_drivers       140
Naive                      137
lgbm_l1_std_categorical    117
lgbm_horizon_safe          114
lgbm_direct                 87
lgbm_l1_std                 84
lgbm_global_minmax          83
lgbm_quantile60_std         72
lgbm_tweedie_raw            68
lgbm_categorical            63
lgbm_hist_drivers           52
lgbm_baseline               42
lgbm_raw                    40
lgbm_dept_raw               39
lgbm_dept_std               38
lgbm_velocity_std           35
Name: count, dtype: int64

Apply the choice at the final cutoff and score it with the same leaderboard as every other model.

In [8]:
final_rows = candidates[candidates.cutoff == TEST_CUTOFF]
by_hand = final_rows[final_rows.model == final_rows.unique_id.map(sku_winner)].assign(model="By_hand_sku")
leaderboard(pd.concat([final_rows[final_rows.model.isin(["lgbm_l1_std", "SeasonalNaive"])], by_hand]), actuals, scope="test")[columns].round(3)

,model,information,coverage,wape,bias,mean_rmsse,median_rmsse,apc
0,lgbm_l1_std,Sales history and calendar only,1.0,0.369,-0.003,1.265,0.671,NaN
1,By_hand_sku,Sales history and calendar only,1.0,0.391,0.042,1.313,0.695,NaN
2,SeasonalNaive,Sales history and calendar only,1.0,0.438,-0.037,1.525,0.929,NaN


Choosing a model for every SKU gives 0.391. That is better than seasonal naive (0.438) but worse than using `lgbm_l1_std` for every product (0.369). Every SKU got the model that was best on its own history, yet together they lose to one good model used everywhere. The rest of the notebook asks why, and what works better.

## 3. Selection at every decision date

The by-hand version chose once, at the final cutoff. To test a selection policy honestly we must also apply it at earlier dates, as it would have run in production. On each decision date it may use only the development runs whose whole 14-week horizon had been observed by then (`completed_runs`). On 2025-01-01 that is the three runs through 2024-09-11; at the final cutoff it is all seven.

A selection policy has two choices:

- **Level**: which series share one choice. One model for the whole portfolio, one per department, class or subclass, or one per SKU.
- **Cost**: how a candidate is scored within a group. By default, its MAE pooled over every SKU-week in the group.

A series with no evidence, or whose chosen model has no forecast at that cutoff, falls back to the portfolio winner. Exact ties go to the first model alphabetically.

`hierarchy` gives each SKU its department, class and subclass. `selection_evidence` collects each candidate pool's development forecasts, on the rows every candidate in the pool forecasts.

In [9]:
hier = hierarchy(panel)
pools = {"stats": stats, "ml": ml, "stats+ml": stats + ml, "stats+ml+future": stats + ml + future}
evidence = {name: selection_evidence(candidates, actuals, models) for name, models in pools.items()}
hier.head(3)

,portfolio,department,class,subclass,sku
unique_id,,,,,
10006_S1,all,Canned Seafood,117.0,1394.0,10006_S1
10008_S1,all,Baby,383.0,1402.0,10008_S1
10010_S1,all,Packaged,226.0,1022.0,10010_S1


### One decision, step by step

Department-level selection on 2025-01-01, with the three runs completed by then: the same `evaluate` and `idxmin` as in section 2, with the department as the series id.

In [10]:
decision = pd.Timestamp("2025-01-01")
runs = completed_runs(decision)
rows = evidence["stats+ml"][evidence["stats+ml"].cutoff.isin(runs)].assign(group=lambda d: d.unique_id.map(hier["department"]))
department_mae = evaluate(to_wide(rows, extra=["group"]), metrics=[mae], models=sorted(stats + ml), id_col="group").drop(columns="metric").set_index("group")
department_winner = department_mae.idxmin(axis=1)
department_winner.value_counts().head(5)

lgbm_l1_std                24
lgbm_l1_hist_drivers       23
lgbm_l1_raw                17
SeasonalNaive              14
lgbm_l1_std_categorical    10
Name: count, dtype: int64

`choose` does the same for any level and date and adds the fallback. It returns the chosen model for every SKU, and the portfolio winner:

In [11]:
department_choice, portfolio_winner = choose(evidence["stats+ml"], hier, runs, level="department")
pd.Series({"departments": len(department_winner), "distinct winners": department_winner.nunique(), "portfolio winner": portfolio_winner,
           "SKUs that keep their department's winner": (department_choice == hier["department"].map(department_winner)).mean()})

departments                                                  126
distinct winners                                              16
portfolio winner                            lgbm_l1_hist_drivers
SKUs that keep their department's winner                     1.0
dtype: object

`strategy` repeats `choose` at every decision date and collects the chosen forecasts as a new model, `average` does the same for an equal-weight average, and `top3` returns the three candidates with the lowest WAPE on the completed runs.

## 4. Selection policies on the development runs

**Question.** Which selection level works best, and does selection beat averaging?

**What changes.** Only the policy: the candidate pool and the level. Every policy runs in both scenarios: `Select_all_*` and `Average_top3_ml` choose from the pool without future drivers, `Select_future_*` and `Average_top3_future` from the pool with them. `Select_stats_sku` and `Select_ml_sku` show what each half of the pool does on its own.

**Why it might help.** Products differ: a model that is best overall may not be best for every product, and finer levels can follow that. Averages smooth out the individual models' errors.

Decisions are the development cutoffs with at least one completed run: 2024-11-06, 2024-12-04 and 2025-01-01, with one, two and three evidence runs.

In [12]:
decisions = [d for d in DEV_CUTOFFS if completed_runs(d)]
policies = {
    "Select_stats_sku": ("stats", "sku"),
    "Select_ml_sku": ("ml", "sku"),
    "Select_all_portfolio": ("stats+ml", "portfolio"),
    "Select_all_department": ("stats+ml", "department"),
    "Select_all_sku": ("stats+ml", "sku"),
    "Select_future_portfolio": ("stats+ml+future", "portfolio"),
    "Select_future_department": ("stats+ml+future", "department"),
    "Select_future_sku": ("stats+ml+future", "sku"),
}


def build(decisions):
    out = [strategy(name, candidates, evidence[pool], hier, decisions, level=level) for name, (pool, level) in policies.items()]
    for d in decisions:
        runs = completed_runs(d)
        out.append(average("Average_stats", candidates, stats, [d]))
        out.append(average("Average_all", candidates, stats + ml, [d]))
        out.append(average("Average_top3_ml", candidates, top3(candidates, actuals, ml, runs), [d]))
        out.append(average("Average_top3_future", candidates, top3(candidates, actuals, ml + future, runs), [d]))
    return pd.concat(out)


dev_strategies = build(decisions)

Strategies and individual candidates scored together on the same product-weeks; the eight best of each information set.

In [13]:
individual = candidates[candidates.model.isin(stats + ml + future)]
dev_compare = leaderboard(pd.concat([individual, dev_strategies]), actuals, scope=decisions)
dev_compare[columns].groupby(dev_compare.information_tier, sort=False).head(8).round(3)

,model,information,coverage,wape,bias,mean_rmsse,median_rmsse,apc
0,lgbm_l1_std_categorical,Sales history and calendar only,1.000,0.357,-0.052,1.479,0.795,0.157
1,Average_stats,Sales history and calendar only,1.000,0.366,-0.059,1.431,0.814,0.140
2,Select_stats_sku,Sales history and calendar only,1.000,0.379,-0.044,1.514,0.852,0.167
3,SeasonalNaive,Sales history and calendar only,0.997,0.380,-0.056,1.601,1.011,0.000
4,lgbm_l1_std,Sales history and calendar only,1.000,0.385,-0.007,1.484,0.809,0.198
5,lgbm_direct,Sales history and calendar only,1.000,0.394,-0.039,1.520,0.837,0.143
6,lgbm_categorical,Sales history and calendar only,1.000,0.414,0.100,1.529,0.851,0.150
7,lgbm_l1_raw,Sales history and calendar only,1.000,0.415,-0.093,1.517,0.826,0.249
19,Select_all_portfolio,Historical exogenous features,1.000,0.356,-0.055,1.480,0.796,0.152
20,lgbm_l1_hist_drivers,Historical exogenous features,1.000,0.356,-0.055,1.480,0.796,0.152


- **Without future drivers**, portfolio selection picks `lgbm_l1_hist_drivers` (0.356). SKU selection (0.360) and the top-three average (0.361) are close behind, and department selection is worse (0.377).
- **With future drivers**, the ranking is the same: portfolio selection 0.291, top-three average 0.293, department 0.302, SKU 0.309.
- These decisions rest on one to three evidence runs, so differences of a point or two are noise. The final window in section 5 is the stronger test.

## 5. Final 14 weeks

The policies are now fixed and applied once at the final cutoff, with all seven development runs as evidence. They are also applied at 2025-02-12 and 2025-03-12, each with the runs completed by then, so that forecast stability (APC) can be measured.

This window was inspected while building these notebooks, so treat the numbers as an exploratory benchmark rather than an untouched holdout.

In [14]:
targets = [TEST_CUTOFF] + list(STABILITY_CUTOFFS)
strategies = build(targets)

members = stats + ["lgbm_baseline", dev_best, dev_best_future]
test_board = leaderboard(pd.concat([individual[individual.model.isin(members)], strategies]), actuals, scope="test")
test_board[columns].round(3)

,model,information,coverage,wape,bias,mean_rmsse,median_rmsse,apc
0,Average_stats,Sales history and calendar only,1.0,0.384,0.006,1.275,0.726,0.109
1,lgbm_baseline,Sales history and calendar only,1.0,0.419,0.109,1.302,0.733,0.092
2,Select_stats_sku,Sales history and calendar only,1.0,0.425,0.029,1.371,0.737,0.090
3,HistoricAverage,Sales history and calendar only,1.0,0.426,-0.011,1.367,0.739,0.010
4,SeasonalNaive,Sales history and calendar only,1.0,0.438,-0.037,1.525,0.929,0.000
5,Naive,Sales history and calendar only,1.0,0.470,0.066,1.332,0.771,0.327
6,Select_all_portfolio,Historical exogenous features,1.0,0.349,-0.028,1.255,0.671,0.107
7,lgbm_l1_hist_drivers,Historical exogenous features,1.0,0.349,-0.028,1.255,0.671,0.107
8,Average_top3_ml,Historical exogenous features,1.0,0.350,-0.017,1.251,0.661,0.105
9,Select_all_department,Historical exogenous features,1.0,0.368,-0.001,1.270,0.678,0.095


### Does selection beat its own best candidate?

Each strategy against the best individual model of its scenario, chosen on development WAPE: `dev_best` without future drivers, `dev_best_future` with them. Negative changes are improvements.

In [15]:
board = test_board.set_index("model")
own_best = {name: dev_best_future if pool == "stats+ml+future" else dev_best for name, (pool, _) in policies.items()}
own_best |= {"Average_stats": dev_best, "Average_all": dev_best, "Average_top3_ml": dev_best, "Average_top3_future": dev_best_future}
metrics = ["wape", "bias", "mean_rmsse", "apc"]
impact = board.loc[list(own_best), metrics]
change = impact - board.loc[list(own_best.values()), metrics].set_axis(impact.index)
pd.concat({"level": impact, "change vs best individual": change}, axis=1).assign(best_individual=pd.Series(own_best)).round(3)

level                           \
                           wape   bias mean_rmsse    apc   
model                                                      
Select_stats_sku          0.425  0.029      1.371  0.090   
Select_ml_sku             0.373  0.013      1.291  0.141   
Select_all_portfolio      0.349 -0.028      1.255  0.107   
Select_all_department     0.368 -0.001      1.270  0.095   
Select_all_sku            0.391  0.042      1.313  0.111   
Select_future_portfolio   0.268  0.023      1.155  0.093   
Select_future_department  0.289  0.048      1.164  0.124   
Select_future_sku         0.298  0.056      1.204  0.101   
Average_stats             0.384  0.006      1.275  0.109   
Average_all               0.374  0.070      1.854  0.091   
Average_top3_ml           0.350 -0.017      1.251  0.105   
Average_top3_future       0.267  0.043      1.138  0.075   

                         change vs best individual                           \
                                              wape   bias mean_rmsse    apc   
model                                                                         
Select_stats_sku                             0.076  0.058      0.116 -0.017   
Select_ml_sku                                0.024  0.041      0.036  0.034   
Select_all_portfolio                         0.000  0.000      0.000  0.000   
Select_all_department                        0.018  0.028      0.015 -0.013   
Select_all_sku                               0.042  0.070      0.058  0.004   
Select_future_portfolio                      0.000  0.000      0.000  0.000   
Select_future_department                     0.021  0.025      0.009  0.031   
Select_future_sku                            0.031  0.034      0.049  0.008   
Average_stats                                0.035  0.034      0.020  0.002   
Average_all                                  0.025  0.098      0.599 -0.016   
Average_top3_ml                              0.001  0.011     -0.004 -0.003   
Average_top3_future                         -0.001  0.021     -0.018 -0.018   

                                      best_individual  
                                                       
model                                                  
Select_stats_sku                 lgbm_l1_hist_drivers  
Select_ml_sku                    lgbm_l1_hist_drivers  
Select_all_portfolio             lgbm_l1_hist_drivers  
Select_all_department            lgbm_l1_hist_drivers  
Select_all_sku                   lgbm_l1_hist_drivers  
Select_future_portfolio   lgbm_l1_future_price_levels  
Select_future_department  lgbm_l1_future_price_levels  
Select_future_sku         lgbm_l1_future_price_levels  
Average_stats                    lgbm_l1_hist_drivers  
Average_all                      lgbm_l1_hist_drivers  
Average_top3_ml                  lgbm_l1_hist_drivers  
Average_top3_future       lgbm_l1_future_price_levels

For context, the five best individual models of each information set on the same window:

In [16]:
rank = leaderboard(individual, actuals, scope="test")
rank[columns].groupby(rank.information_tier, sort=False).head(5).round(3)

,model,information,coverage,wape,bias,mean_rmsse,median_rmsse,apc
0,lgbm_l1_std_categorical,Sales history and calendar only,1.0,0.350,-0.020,1.257,0.667,0.113
1,lgbm_l1_std,Sales history and calendar only,1.0,0.369,-0.003,1.265,0.671,0.118
2,lgbm_l1_raw,Sales history and calendar only,1.0,0.375,-0.073,1.229,0.685,0.166
3,lgbm_direct,Sales history and calendar only,1.0,0.380,-0.033,1.301,0.694,0.128
4,lgbm_categorical,Sales history and calendar only,1.0,0.400,0.105,1.270,0.703,0.107
17,lgbm_l1_hist_drivers,Historical exogenous features,1.0,0.349,-0.028,1.255,0.671,0.107
18,lgbm_hist_drivers,Historical exogenous features,1.0,0.396,0.107,1.266,0.711,0.102
19,lgbm_l1_future_price_levels,Full future-driver visibility,1.0,0.268,0.023,1.155,0.568,0.093
20,lgbm_future_discount,Full future-driver visibility,1.0,0.274,0.049,1.148,0.572,0.086
21,lgbm_future_price_index,Full future-driver visibility,1.0,0.276,0.052,1.147,0.571,0.084


**Without future drivers.** Portfolio selection chooses the best individual model, `lgbm_l1_hist_drivers` (0.349). The top-three average is level with it (0.350). Finer selection is worse: department 0.368, SKU 0.391. Choosing among the statistical baselines alone gives 0.425.

**With future drivers.** Portfolio selection chooses `lgbm_l1_future_price_levels` (0.268). The top-three average is level on WAPE (0.267) and revises less between origins (APC 0.075 against 0.093). Department (0.289) and SKU (0.298) selection are worse again.

**Between the scenarios.** For the same policy the plan is worth about 8 WAPE points (0.349 against 0.268), far more than any choice of selection method. Future-driver scores assume the plan is carried out as recorded.

## Takeaways

- **Pick the best model on backtests, at the portfolio level.** Finer selection follows noise: with a few runs per SKU, the "best" model for a product is often luck. Optional section B shows that the winner changes between runs for most SKUs.
- **A top-three average is as accurate as the best model** and can revise less. It is a reasonable default when forecast stability matters.
- **Information beats method.** The price and promotion plan moves accuracy far more than any selection or averaging policy.

---

# Optional advanced experiments

Sections 1 to 5 are the workshop narrative. Sections A to F test the conclusions further; read them from the saved outputs, or run them after the session. They run in order, because each reuses strategies from the one before (D, the hindsight bound, stands alone), and the last cell, which saves the results for notebook 08, needs all of them. Notebook 08 reads the saved file in the repository, so there is no need to run these sections to continue.

| section | question | run time |
|---|---|---|
| A | selection level and cost across the hierarchy | about 2 min |
| B | penalties on SKU selection; how stable SKU winners are | under 1 min |
| C | does a larger candidate pool help selection? | about 11 min |
| D | hindsight bound on SKU selection | seconds |
| E | an ensemble as one more candidate | about 3.5 min |
| F | ensemble diversity | about 1.5 min |

## A. Selection level and cost across the hierarchy

**Question.** Section 4 tried portfolio, department and SKU. Is there a level in between that works, and does the way we score candidates matter?

**What changes.** Five levels (portfolio, department, class, subclass, SKU) and three costs, always with the candidates without future drivers:

- **pooled**: MAE of the SKU forecasts, pooled over every SKU-week in the group (the default).
- **aggregated**: actuals and forecasts summed to group-week first, then MAE. The same as pooled for single-SKU groups, so skipped at SKU level.
- **mae_bias**: pooled MAE plus the absolute mean error.

**Why it might help.** Coarser groups give each choice more evidence; finer groups can follow differences between products. Aggregated cost and a bias penalty reward models that get the total right.

SKUs with a missing hierarchy value form an `Unknown` group at that level (three SKUs). The pooled portfolio, department and SKU policies are those of section 4 and keep their names.

In [17]:
levels = LEVELS
costs = {"pooled": {}, "aggregated": dict(aggregate="hierarchy"), "mae_bias": dict(lam=1.0)}
live = {"portfolio": "Select_all_portfolio", "department": "Select_all_department", "sku": "Select_all_sku"}

grid = {}
for level in levels:
    for cost, options in costs.items():
        if cost == "aggregated" and level == "sku":
            continue
        name = live.get(level, f"Select_{level}_pooled") if cost == "pooled" else f"Select_{level}_{cost}"
        grid[name] = dict(level=level, **options)

new_policies = {name: policy for name, policy in grid.items() if name not in policies}
hier_runs = pd.concat([strategy(name, candidates, evidence["stats+ml"], hier, decisions + targets, **policy) for name, policy in new_policies.items()])
grid_runs = pd.concat([dev_strategies[dev_strategies.model.isin(grid)], strategies[strategies.model.isin(grid)], hier_runs])

Final 14 weeks, chosen with all seven development runs. `n_groups` counts the groups at that level and `n_models` the distinct models chosen, including the portfolio fallback.

In [18]:
choices = {name: choose(evidence["stats+ml"], hier, completed_runs(TEST_CUTOFF), **policy)[0] for name, policy in grid.items()}
sizes = pd.DataFrame({name: {"n_groups": hier[policy["level"]].nunique(), "n_models": choices[name].nunique()} for name, policy in grid.items()}).T

hier_test = leaderboard(pd.concat([individual[individual.model.isin(members)], grid_runs]), actuals, scope="test").set_index("model")
hier_test.loc[list(grid), ["wape", "bias", "mean_rmsse", "median_rmsse", "apc", "coverage"]].join(sizes).sort_values("wape").round(3)

,wape,bias,mean_rmsse,median_rmsse,apc,coverage,n_groups,n_models
model,,,,,,,,
Select_all_portfolio,0.349,-0.028,1.255,0.671,0.107,1.0,1,1
Select_all_department,0.368,-0.001,1.270,0.678,0.095,1.0,126,16
Select_portfolio_mae_bias,0.369,-0.003,1.265,0.671,0.149,1.0,1,1
Select_class_pooled,0.372,0.007,1.296,0.701,0.102,1.0,345,19
Select_subclass_mae_bias,0.373,0.059,1.294,0.721,0.141,1.0,773,19
Select_subclass_pooled,0.380,0.031,1.445,0.709,0.118,1.0,773,19
Select_department_mae_bias,0.384,0.033,1.329,0.720,0.127,1.0,126,19
Select_class_mae_bias,0.385,0.058,1.322,0.735,0.124,1.0,345,19
Select_subclass_aggregated,0.387,0.041,1.482,0.731,0.119,1.0,773,19


Is the ranking stable? WAPE at all six decision dates (three development decisions with one to three evidence runs; the final and two stability cutoffs with four to seven), and each strategy's rank within each date.

In [19]:
wape = wape_by_cutoff(grid_runs, actuals, decisions + targets)
ranks = wape.rank()
stability = pd.DataFrame({"mean_wape": wape.mean(axis=1), "mean_rank": ranks.mean(axis=1), "best_rank": ranks.min(axis=1), "worst_rank": ranks.max(axis=1),
                          "wape_final": wape[TEST_CUTOFF], "rank_final": ranks[TEST_CUTOFF]})
stability.sort_values("mean_rank").round(3)

,mean_wape,mean_rank,best_rank,worst_rank,wape_final,rank_final
model,,,,,,
Select_all_portfolio,0.354,1.250,1.0,2.0,0.349,1.0
Select_subclass_pooled,0.369,4.500,1.0,8.0,0.380,6.0
Select_all_sku,0.372,5.333,3.0,10.0,0.391,10.0
Select_class_pooled,0.373,5.667,4.0,8.0,0.372,4.0
Select_all_department,0.373,5.833,2.0,14.0,0.368,2.0
Select_subclass_aggregated,0.378,7.000,4.0,12.0,0.387,9.0
Select_portfolio_mae_bias,0.384,7.250,1.5,13.0,0.369,3.0
Select_sku_mae_bias,0.381,7.500,4.0,13.0,0.396,13.0
Select_subclass_mae_bias,0.382,7.833,5.0,10.0,0.373,5.0


Change in WAPE points against pooled portfolio selection at each date; `wins` counts the dates where the policy is better.

In [20]:
(wape.sub(wape.loc["Select_all_portfolio"], axis=1) * 100).round(2).loc[list(grid)].assign(wins=lambda d: (d < 0).sum(axis=1))

cutoff,2024-11-06 00:00:00,2024-12-04 00:00:00,2025-01-01 00:00:00,2025-02-12 00:00:00,2025-03-12 00:00:00,2025-04-09 00:00:00,wins
model,,,,,,,
Select_all_portfolio,0.00,0.00,0.00,0.00,0.00,0.00,0
Select_portfolio_aggregated,0.17,6.56,8.42,11.38,8.53,8.86,0
Select_portfolio_mae_bias,0.00,4.81,4.97,4.70,1.20,1.97,0
Select_all_department,3.03,3.00,0.41,1.70,1.47,1.85,0
Select_department_aggregated,1.05,2.10,4.34,4.80,3.73,4.39,0
Select_department_mae_bias,2.82,2.72,3.52,5.61,4.07,3.53,0
Select_class_pooled,1.34,2.65,1.42,2.03,1.57,2.27,0
Select_class_aggregated,1.96,3.45,3.30,4.28,3.84,4.64,0
Select_class_mae_bias,1.85,4.11,2.05,4.21,4.78,3.60,0


- **Portfolio selection is best**: 0.349 in the final window, and the best mean rank over the six decision dates (1.25, never worse than second).
- **Every finer level is worse in the final window**: department 0.368, class 0.372, subclass 0.380, SKU 0.391. Against portfolio selection they lose at almost every date.
- **Aggregated cost is worse than pooled at every level.** Summing to group-week rewards models that get the group total right and hides the SKU-level errors that WAPE counts.
- **The bias penalty does not help reliably.** It is worse at every level except subclass in the final window (0.373 against 0.380), and over the six dates that reverses (mean 0.382 against 0.369).
- Six overlapping windows are not independent evidence. Treat differences of a point or less as noise.

## B. Penalties on SKU-level selection

**Question.** SKU-level selection follows noise. Can a penalty in the cost make it more careful?

**What changes.** Three options of `choose`, one at a time, on the development decisions:

- `lam` adds the absolute mean error: `cost = MAE + lam * |mean error|`.
- `margin`: a SKU leaves the portfolio winner only if its own winner lowers the cost by at least this fraction.
- `mu` adds the candidate's revision size between consecutive evidence runs, in units of demand (`mu * APC * mean(actual)`).

**Why it might help.** A bias penalty favors models that get the level right, a margin keeps a SKU on the proven model unless the evidence is clear, and a revision penalty favors stable forecasts.

In [21]:
variants = {f"lam={lam}": dict(level="sku", lam=lam) for lam in [0, 1, 4]}
variants |= {f"margin={m}": dict(level="sku", margin=m) for m in [0.05, 0.2]}
variants |= {f"mu={mu}": dict(level="sku", mu=mu) for mu in [1, 4]}
runs = pd.concat([strategy(name, candidates, evidence["stats+ml"], hier, decisions, **policy) for name, policy in variants.items()])
leaderboard(runs, actuals, scope=decisions)[columns + ["apc_median"]].round(3)

,model,information,coverage,wape,bias,mean_rmsse,median_rmsse,apc,apc_median
0,margin=0.2,Historical exogenous features,1.0,0.351,-0.049,1.459,0.809,0.162,0.164
1,margin=0.05,Historical exogenous features,1.0,0.354,-0.024,1.472,0.819,0.179,0.176
2,lam=0,Historical exogenous features,1.0,0.360,-0.013,1.478,0.829,0.190,0.179
3,mu=1,Historical exogenous features,1.0,0.361,-0.032,1.501,0.834,0.177,0.146
4,lam=1,Historical exogenous features,1.0,0.368,-0.004,1.488,0.835,0.202,0.194
5,mu=4,Historical exogenous features,1.0,0.372,-0.050,1.516,0.861,0.193,0.152
6,lam=4,Historical exogenous features,1.0,0.388,0.016,1.501,0.850,0.222,0.198


Only the **margin** helps: with 0.2, SKU selection reaches 0.351, against 0.360 without it and 0.356 for portfolio selection, and its forecasts revise less. It works by keeping most SKUs on the portfolio winner, so it moves SKU selection towards portfolio selection. The bias penalty (`lam`) and the revision penalty (`mu`) make accuracy worse. This sweep covers only the three development decisions, with one to three runs each, so it is weak evidence and was not carried to the final window.

### How stable are SKU winners?

The best candidate for each SKU in each development run, and how often it changes from one run to the next.

In [22]:
per_run = evidence["stats+ml"].assign(abs_error=lambda d: (d.yhat - d.y).abs()).groupby(["unique_id", "cutoff", "model"]).abs_error.mean().unstack("model")
winner = per_run.idxmin(axis=1).unstack("cutoff")
changed = (winner.iloc[:, 1:].to_numpy() != winner.iloc[:, :-1].to_numpy()).mean()
print(f"SKU winner differs between consecutive development runs for {changed:.0%} of SKU-run pairs")
print(f"Distinct winners per SKU across {winner.shape[1]} runs: mean {winner.nunique(axis=1).mean():.1f}")

SKU winner differs between consecutive development runs for 72% of SKU-run pairs
Distinct winners per SKU across 7 runs: mean 4.6


A SKU's winner changes between consecutive runs for 72% of SKU-run pairs, so most of the per-SKU choice is noise. That is why SKU-level selection loses to the portfolio choice.

## C. Growing the candidate pool

**Question.** Does selection get better as we give it more, and better, candidates?

**What changes.** The candidate pool. Eight nested pools of saved forecasts, each containing the previous one:

| Pool | Adds |
| --- | --- |
| P1_stats | Naive, HistoricAverage, SeasonalNaive |
| P2_basic | basic LightGBM from notebook 02: raw and standardized target |
| P3_scaled_direct | min-max scaling (04) and direct and horizon-safe forecasting (05) |
| P4_objectives_pooling | department and velocity models (04), other objectives (06) |
| P5_static | department, class and subclass as categorical features (03, and its L1 version from 06) |
| P6_hist_drivers | historical price and promotion drivers (03, and its L1 version from 06) |
| P7_behavior | the historical behavior models of notebook 03c |
| P8_future | the future-driver models of notebooks 03, 03b and 06 |

P6 is the pool without future drivers used above. P1 to P7 need no future drivers; P8 needs the plan, so its strategies belong to the second scenario.

**Why it might help.** More candidates give each group more chances to find a good model, but also more chances to pick one that was lucky on the evidence.

In [23]:
behavior_forecasts = load_forecasts(["03c_behavior_features"])
behavior = sorted(behavior_forecasts.model.unique())
candidates = pd.concat([candidates, behavior_forecasts])

added = {
    "P1_stats": stats,
    "P2_basic": ["lgbm_raw", "lgbm_baseline"],
    "P3_scaled_direct": ["lgbm_global_minmax", "lgbm_direct", "lgbm_horizon_safe"],
    "P5_static": ["lgbm_categorical", "lgbm_l1_std_categorical"],
    "P6_hist_drivers": ["lgbm_hist_drivers", "lgbm_l1_hist_drivers"],
}
added["P4_objectives_pooling"] = [m for m in stats + ml if m not in sum(added.values(), [])]
added = {pool: added[pool] for pool in ["P1_stats", "P2_basic", "P3_scaled_direct", "P4_objectives_pooling", "P5_static", "P6_hist_drivers"]}
added["P7_behavior"] = [m for m in behavior if m in usable_models(behavior_forecasts, future_drivers=False)]
added["P8_future"] = future

nested = {}
for pool, models in added.items():
    nested[pool] = sum(list(added.values())[:len(nested) + 1], [])
assert sorted(nested["P6_hist_drivers"]) == sorted(stats + ml)

pools |= nested
evidence |= {pool: selection_evidence(candidates, actuals, models) for pool, models in nested.items()}

info = candidates.drop_duplicates("model").set_index("model")
pd.DataFrame({
    "adds": {pool: ", ".join(models) for pool, models in added.items()},
    "information": {pool: ", ".join(sorted(set(info.information_tier[models]))) for pool, models in added.items()},
    "pool_size": {pool: len(models) for pool, models in nested.items()},
})

,adds,information,pool_size
P1_stats,"Naive, HistoricAverage, SeasonalNaive",history,3
P2_basic,"lgbm_raw, lgbm_baseline",history,5
P3_scaled_direct,"lgbm_global_minmax, lgbm_direct, lgbm_horizon_...",history,8
P4_objectives_pooling,"lgbm_dept_raw, lgbm_dept_std, lgbm_l1_raw, lgb...",history,15
P5_static,"lgbm_categorical, lgbm_l1_std_categorical",history,17
P6_hist_drivers,"lgbm_hist_drivers, lgbm_l1_hist_drivers",historical_drivers,19
P7_behavior,"lgbm_hist_class_agg, lgbm_hist_combined, lgbm_...",historical_drivers,29
P8_future,"lgbm_adj_calendar_future, lgbm_adj_future, lgb...",future_drivers,37


For every pool and level, the tournament picks each group's winner on the decision date, as before. Two references use the same pool and evidence:

- **Best individual** (`Best_<pool>`): the single pool member with the lowest WAPE on the completed runs.
- **Average** (`Average_<pool>`): the equal-weight mean of all pool members.

Strategies that already exist keep their names: `Select_stats_sku`, the P6 `Select_all_*` and pooled class and subclass strategies, `Average_stats` and `Average_all`.

In [24]:
all_decisions = decisions + targets
reuse = {("P1_stats", "sku"): "Select_stats_sku", ("P6_hist_drivers", "portfolio"): "Select_all_portfolio", ("P6_hist_drivers", "department"): "Select_all_department",
         ("P6_hist_drivers", "class"): "Select_class_pooled", ("P6_hist_drivers", "subclass"): "Select_subclass_pooled", ("P6_hist_drivers", "sku"): "Select_all_sku"}
avg_of = {pool: {"P1_stats": "Average_stats", "P6_hist_drivers": "Average_all"}.get(pool, f"Average_{pool}") for pool in nested}

tour = pd.DataFrame([(reuse.get((pool, level), f"Tour_{pool}_{level}"), pool, level) for pool in nested for level in levels], columns=["model", "pool", "level"])
new = tour[~tour.model.isin(list(grid) + list(policies))]
tour_runs = pd.concat([strategy(row.model, candidates, evidence[row.pool], hier, all_decisions, level=row.level) for row in new.itertuples()])
pool_averages = pd.concat([average(avg_of[pool], candidates, nested[pool], all_decisions) for pool in nested if pool not in ("P1_stats", "P6_hist_drivers")])

best = {(pool, d): best_individual(f"Best_{pool}", candidates, actuals, nested[pool], d) for pool in nested for d in all_decisions}
best_runs = pd.concat([rows for rows, name in best.values()])

pool_runs = pd.concat([dev_strategies, strategies, grid_runs, tour_runs, pool_averages, best_runs])
pool_runs = pool_runs[pool_runs.model.isin(list(tour.model) + list(avg_of.values()) + [f"Best_{pool}" for pool in nested])].drop_duplicates(["unique_id", "cutoff", "ds", "model"])

Final 14 weeks (exploratory). First the references per pool, then the tournament WAPE with pools as rows and levels as columns.

In [25]:
def matrix(values):
    """Pool x level table from a Series indexed by strategy name."""
    return tour.assign(value=tour.model.map(values)).pivot(index="pool", columns="level", values="value").loc[list(nested), levels]


pool_board = leaderboard(pool_runs, actuals, scope="test").set_index("model")
references = pd.DataFrame({
    "candidates": {pool: len(nested[pool]) for pool in nested},
    "best_individual": {pool: best[(pool, TEST_CUTOFF)][1] for pool in nested},
    "best_wape": {pool: pool_board.wape[f"Best_{pool}"] for pool in nested},
    "average_wape": {pool: pool_board.wape[avg_of[pool]] for pool in nested},
})
display(references.round(3))
matrix(pool_board.wape).round(3)

,candidates,best_individual,best_wape,average_wape
P1_stats,3,SeasonalNaive,0.438,0.384
P2_basic,5,SeasonalNaive,0.438,0.383
P3_scaled_direct,8,lgbm_direct,0.380,0.390
P4_objectives_pooling,15,lgbm_l1_std,0.369,0.382
P5_static,17,lgbm_l1_std_categorical,0.350,0.378
P6_hist_drivers,19,lgbm_l1_hist_drivers,0.349,0.374
P7_behavior,29,lgbm_l1_hist_drivers,0.349,0.377
P8_future,37,lgbm_l1_future_price_levels,0.268,0.349


level,portfolio,department,class,subclass,sku
pool,,,,,
P1_stats,0.438,0.430,0.416,0.422,0.425
P2_basic,0.438,0.434,0.417,0.429,0.425
P3_scaled_direct,0.380,0.425,0.423,0.430,0.417
P4_objectives_pooling,0.369,0.379,0.376,0.388,0.399
P5_static,0.350,0.369,0.372,0.380,0.391
P6_hist_drivers,0.349,0.368,0.372,0.380,0.391
P7_behavior,0.349,0.366,0.371,0.379,0.388
P8_future,0.268,0.289,0.306,0.297,0.298


Change in WAPE points against the best individual model and against the pool average. Negative means the tournament forecasts better.

In [26]:
vs_best = matrix(pool_board.wape).sub(references.best_wape, axis=0) * 100
vs_average = matrix(pool_board.wape).sub(references.average_wape, axis=0) * 100
pd.concat({"vs best individual": vs_best, "vs pool average": vs_average}, axis=1).round(2)

vs best individual                                  \
level                          portfolio department class subclass   sku   
pool                                                                       
P1_stats                             0.0      -0.75 -2.14    -1.57 -1.24   
P2_basic                             0.0      -0.39 -2.08    -0.87 -1.26   
P3_scaled_direct                     0.0       4.48  4.26     4.99  3.71   
P4_objectives_pooling                0.0       1.03  0.69     1.95  3.06   
P5_static                            0.0       1.83  2.13     2.96  4.10   
P6_hist_drivers                      0.0       1.85  2.27     3.05  4.16   
P7_behavior                          0.0       1.67  2.15     2.96  3.85   
P8_future                            0.0       2.09  3.87     2.95  3.04   

                      vs pool average                                  
level                       portfolio department class subclass   sku  
pool                                                                   
P1_stats                         5.39       4.63  3.25     3.82  4.14  
P2_basic                         5.45       5.06  3.37     4.58  4.19  
P3_scaled_direct                -0.94       3.54  3.32     4.05  2.77  
P4_objectives_pooling           -1.34      -0.31 -0.65     0.61  1.72  
P5_static                       -2.72      -0.89 -0.59     0.24  1.38  
P6_hist_drivers                 -2.51      -0.66 -0.24     0.54  1.65  
P7_behavior                     -2.75      -1.08 -0.61     0.21  1.10  
P8_future                       -8.10      -6.01 -4.23    -5.15 -5.06

Bias, mean RMSSE, APC and coverage from the same leaderboard.

In [27]:
pd.concat({metric: matrix(pool_board[metric]) for metric in ["bias", "mean_rmsse", "apc", "coverage"]}, axis=1).round(3)

bias                                   mean_rmsse  \
level                 portfolio department  class subclass    sku  portfolio   
pool                                                                           
P1_stats                 -0.037      0.007  0.026    0.045  0.029      1.525   
P2_basic                 -0.037      0.055  0.081    0.101  0.091      1.525   
P3_scaled_direct         -0.033      0.047  0.083    0.098  0.078      1.301   
P4_objectives_pooling    -0.003      0.000 -0.001    0.024  0.042      1.265   
P5_static                -0.020      0.003  0.006    0.028  0.044      1.257   
P6_hist_drivers          -0.028     -0.001  0.007    0.031  0.042      1.255   
P7_behavior              -0.028     -0.002  0.008    0.031  0.044      1.255   
P8_future                 0.023      0.048  0.048    0.043  0.056      1.155   

                                                              apc             \
level                 department  class subclass    sku portfolio department   
pool                                                                           
P1_stats                   1.389  1.378    1.376  1.371     0.000      0.083   
P2_basic                   1.404  1.463    1.483  1.346     0.000      0.080   
P3_scaled_direct           1.362  1.377    1.480  1.333     0.131      0.091   
P4_objectives_pooling      1.278  1.301    1.448  1.319     0.119      0.143   
P5_static                  1.274  1.298    1.449  1.315     0.114      0.104   
P6_hist_drivers            1.270  1.296    1.445  1.313     0.109      0.097   
P7_behavior                1.266  1.295    1.438  1.309     0.109      0.098   
P8_future                  1.162  1.195    1.196  1.203     0.094      0.124   

                                              coverage                   \
level                  class subclass    sku portfolio department class   
pool                                                                      
P1_stats               0.076    0.098  0.090       1.0        1.0   1.0   
P2_basic               0.095    0.111  0.109       1.0        1.0   1.0   
P3_scaled_direct       0.102    0.110  0.106       1.0        1.0   1.0   
P4_objectives_pooling  0.116    0.121  0.116       1.0        1.0   1.0   
P5_static              0.104    0.121  0.115       1.0        1.0   1.0   
P6_hist_drivers        0.103    0.119  0.112       1.0        1.0   1.0   
P7_behavior            0.101    0.118  0.108       1.0        1.0   1.0   
P8_future              0.096    0.105  0.102       1.0        1.0   1.0   

                                     
level                 subclass  sku  
pool                                 
P1_stats                   1.0  1.0  
P2_basic                   1.0  1.0  
P3_scaled_direct           1.0  1.0  
P4_objectives_pooling      1.0  1.0  
P5_static                  1.0  1.0  
P6_hist_drivers            1.0  1.0  
P7_behavior                1.0  1.0  
P8_future                  1.0  1.0

Distinct models chosen at each level at the final decision (including the portfolio fallback), and the number of groups per level.

In [28]:
chosen = {row.model: choose(evidence[row.pool], hier, completed_runs(TEST_CUTOFF), level=row.level)[0] for row in tour.itertuples()}
winners = matrix(pd.Series({model: choice.nunique() for model, choice in chosen.items()}))
groups = pd.DataFrame({"groups": {level: hier[level].nunique() for level in levels}}).T
pd.concat([winners.assign(candidates=[len(nested[pool]) for pool in nested]), groups.assign(candidates="")])

,portfolio,department,class,subclass,sku,candidates
P1_stats,1,3,3,3,3,3
P2_basic,1,5,5,5,5,5
P3_scaled_direct,1,8,8,8,8,8
P4_objectives_pooling,1,14,15,15,15,15
P5_static,1,14,17,17,17,17
P6_hist_drivers,1,16,19,19,19,19
P7_behavior,1,20,29,28,29,29
P8_future,1,24,33,36,37,37
groups,1,126,345,773,2000,


Consistency across the six decision dates: how often each tournament beats the best individual and the pool average, and its mean change against the best individual, separately for the development decisions and the later cutoffs.

In [29]:
pool_wape = wape_by_cutoff(pool_runs, actuals, all_decisions)
best_wape = pool_wape.loc[[f"Best_{pool}" for pool in tour.pool]].set_axis(tour.model)
average_wape = pool_wape.loc[[avg_of[pool] for pool in tour.pool]].set_axis(tour.model)
delta = pool_wape.loc[tour.model] - best_wape

pd.concat({
    "beats best individual": matrix((pool_wape.loc[tour.model] < best_wape).sum(axis=1)),
    "beats pool average": matrix((pool_wape.loc[tour.model] < average_wape).sum(axis=1)),
    "change vs best, development": matrix(delta[decisions].mean(axis=1) * 100).round(2),
    "change vs best, later cutoffs": matrix(delta[targets].mean(axis=1) * 100).round(2),
}, axis=1)

beats best individual                                \
level                             portfolio department class subclass sku   
pool                                                                        
P1_stats                                  0          6     5        6   6   
P2_basic                                  0          5     5        6   6   
P3_scaled_direct                          0          0     1        4   4   
P4_objectives_pooling                     0          0     2        2   3   
P5_static                                 0          0     0        1   0   
P6_hist_drivers                           0          0     0        1   0   
P7_behavior                               0          0     1        2   0   
P8_future                                 0          0     0        0   0   

                      beats pool average                                \
level                          portfolio department class subclass sku   
pool                                                                     
P1_stats                               0          0     0        0   0   
P2_basic                               0          0     0        0   2   
P3_scaled_direct                       2          0     0        2   4   
P4_objectives_pooling                  5          2     6        5   4   
P5_static                              6          5     6        5   4   
P6_hist_drivers                        6          4     5        4   4   
P7_behavior                            6          5     5        5   4   
P8_future                              6          6     6        6   6   

                      change vs best, development                            \
level                                   portfolio department class subclass   
pool                                                                          
P1_stats                                      0.0      -2.96 -2.58    -3.37   
P2_basic                                      0.0      -0.64 -2.14    -2.96   
P3_scaled_direct                              0.0       1.98  0.80    -0.52   
P4_objectives_pooling                         0.0       1.20 -0.34    -1.46   
P5_static                                     0.0       3.21  2.02     0.91   
P6_hist_drivers                               0.0       2.15  1.80     0.91   
P7_behavior                                   0.0       2.12  1.27     0.48   
P8_future                                     0.0       1.13  2.10     1.86   

                            change vs best, later cutoffs                   \
level                   sku                     portfolio department class   
pool                                                                         
P1_stats              -4.64                           0.0      -2.41 -4.18   
P2_basic              -4.71                           0.0      -2.86 -4.26   
P3_scaled_direct      -2.04                           0.0       2.10  1.80   
P4_objectives_pooling -1.97                           0.0       1.00  0.79   
P5_static              0.60                           0.0       1.50  1.85   
P6_hist_drivers        0.46                           0.0       1.67  1.96   
P7_behavior            0.46                           0.0       1.58  1.90   
P8_future              1.79                           0.0       1.80  3.02   

                                      
level                 subclass   sku  
pool                                  
P1_stats                 -4.10 -3.74  
P2_basic                 -4.10 -3.88  
P3_scaled_direct          1.58  1.18  
P4_objectives_pooling     1.13  2.09  
P5_static                 2.00  3.02  
P6_hist_drivers           2.03  3.06  
P7_behavior               1.98  2.77  
P8_future                 2.18  2.66

- **The best individual model improves as the pool grows**: seasonal naive (0.438) with P1 and P2, then `lgbm_direct` (0.380), `lgbm_l1_std` (0.369), `lgbm_l1_std_categorical` (0.350) and `lgbm_l1_hist_drivers` (0.349). With the plan (P8) it is `lgbm_l1_future_price_levels` (0.268).
- **Portfolio selection always picks that model.** From P3 on, finer levels are worse than the best individual model by 0.7 to 4.2 points. They help only when the pool holds just the statistical baselines.
- **The behavior models of 03c (P7) add nothing** at portfolio level and at most 0.3 points at finer levels.
- **The plan (P8) moves every policy by about 8 points**, more than any selection choice. The pool average gains even more (0.349 against 0.377), because most of P8's new members are much better than the old ones.

## D. Hindsight bound

**Question.** How much could SKU-level selection gain if it knew the future?

**What changes.** Each SKU's winner is chosen with the final-window actuals, from the candidates without future drivers. This is not achievable, so it is labeled a leakage diagnostic. It shows the ceiling that noisy selection is chasing.

In [30]:
rows = candidates[candidates.model.isin(stats + ml) & (candidates.cutoff == TEST_CUTOFF)]
sku_mae = evaluate(to_wide(rows.merge(actuals, on=["unique_id", "ds"])), metrics=[mae], models=sorted(stats + ml)).set_index("unique_id")
hindsight_pick = sku_mae.drop(columns="metric").idxmin(axis=1)
hindsight = rows[rows.model == rows.unique_id.map(hindsight_pick)].assign(model="Best_in_hindsight_sku", experiment="07_model_selection", information_tier="leakage_diagnostic")

In [31]:
leaderboard(pd.concat([hindsight, strategies[strategies.model.isin(["Select_all_sku", "Select_all_portfolio"])]]), actuals, scope="test")[columns].round(3)

,model,information,coverage,wape,bias,mean_rmsse,median_rmsse,apc
0,Select_all_portfolio,Historical exogenous features,1.0,0.349,-0.028,1.255,0.671,0.111
1,Select_all_sku,Historical exogenous features,1.0,0.391,0.042,1.313,0.695,0.120
2,Best_in_hindsight_sku,Leakage diagnostic,1.0,0.237,-0.060,0.902,0.545,NaN


Choosing each SKU's model with hindsight would cut WAPE from 0.349 to 0.237. Real SKU-level selection reaches 0.391. The potential gain is large, but a few backtest runs cannot identify which SKUs would gain.

## E. Selection with an ensemble as a candidate

**Question.** The top-three average was as good as the best model. If we add it to the pool, can selection improve on it?

**What changes.** `Average_top3_ml` becomes one more candidate, `Ens_top3`, next to the candidates without future drivers, and the tournaments run again at each level. Two policies:

- **Hybrid**: the usual pooled-MAE tournament.
- **Hybrid_margin**: the ensemble is the default; a group switches to an individual model only if that model's cost is at least 5% lower.

**Why it might help.** Groups where one model is clearly better can take it, and the rest keep the ensemble's smoother forecast.

The ensemble also needs forecasts on the evidence runs, and members picked on the same runs would flatter it. So for each evidence run the members are the top three on the *other* completed runs (leave one out). That needs two completed runs, so these comparisons start at the 2024-12-04 decision.

In [32]:
hybrid_decisions = [d for d in all_decisions if len(completed_runs(d)) > 1]
candidates = pd.concat([candidates] + [average("Ens_top3", candidates, top3(candidates, actuals, ml, completed_runs(d)), [d]) for d in hybrid_decisions])

hybrid_policies = {f"Hybrid_{level}": dict(level=level) for level in levels}
hybrid_policies |= {f"Hybrid_margin_{level}": dict(level=level, margin=0.05, default="Ens_top3") for level in levels}

picks = {}
hybrid_runs = []
for d in hybrid_decisions:
    hybrid_evidence = common_rows(pd.concat([evidence["stats+ml"], ensemble_evidence("Ens_top3", candidates, actuals, ml, d)]))
    hybrid_runs += [strategy(name, candidates, hybrid_evidence, hier, [d], picks=picks, **policy) for name, policy in hybrid_policies.items()]
hybrid_runs = pd.concat(hybrid_runs)

Final window: the standalone ensemble and the best individual model, then each policy family by level. The P5 and P6 tournaments from the previous section are the plain-selection references.

In [33]:
standalone = pd.concat([dev_strategies, strategies])
standalone = standalone[standalone.model == "Average_top3_ml"]
rivals = list(tour.model[tour.pool.isin(["P5_static", "P6_hist_drivers"])]) + ["Best_P6_hist_drivers"]
hybrid_all = pd.concat([hybrid_runs, standalone, pool_runs[pool_runs.model.isin(rivals)]])
hybrid_all = hybrid_all[hybrid_all.cutoff.isin(hybrid_decisions)]

families = {"P5 tournament": tour[tour.pool == "P5_static"], "P6 tournament": tour[tour.pool == "P6_hist_drivers"]}
design = pd.concat([f.assign(family=name) for name, f in families.items()]
                   + [pd.DataFrame({"model": [f"{p}_{level}" for level in levels], "level": levels, "family": name}) for p, name in [("Hybrid", "Hybrid"), ("Hybrid_margin", "Hybrid margin")]])


def by_family(values):
    return design.assign(value=design.model.map(values)).pivot(index="family", columns="level", values="value").loc[["P5 tournament", "P6 tournament", "Hybrid", "Hybrid margin"], levels]


hybrid_board = leaderboard(hybrid_all, actuals, scope="test").set_index("model")
display(hybrid_board.loc[["Average_top3_ml", "Best_P6_hist_drivers"], ["wape", "bias", "mean_rmsse", "apc", "coverage"]].round(3))
pd.concat({metric: by_family(hybrid_board[metric]) for metric in ["wape", "bias", "mean_rmsse", "apc", "coverage"]}, axis=1).round(3)

,wape,bias,mean_rmsse,apc,coverage
model,,,,,
Average_top3_ml,0.350,-0.017,1.251,0.107,1.0
Best_P6_hist_drivers,0.349,-0.028,1.255,0.110,1.0


wape                                        bias  \
level         portfolio department  class subclass    sku portfolio   
family                                                                
P5 tournament     0.350      0.369  0.372     0.38  0.391    -0.020   
P6 tournament     0.349      0.368  0.372     0.38  0.391    -0.028   
Hybrid            0.349      0.367  0.373     0.38  0.390    -0.028   
Hybrid margin     0.350      0.370  0.369     0.37  0.388    -0.017   

                                                 ...       apc             \
level         department  class subclass    sku  ... portfolio department   
family                                           ...                        
P5 tournament      0.003  0.006    0.028  0.044  ...     0.116      0.109   
P6 tournament     -0.001  0.007    0.031  0.042  ...     0.110      0.102   
Hybrid             0.001  0.008    0.031  0.041  ...     0.110      0.098   
Hybrid margin      0.012  0.016    0.018  0.044  ...     0.107      0.102   

                                      coverage                                 
level          class subclass    sku portfolio department class subclass  sku  
family                                                                         
P5 tournament  0.111    0.128  0.120       1.0        1.0   1.0      1.0  1.0  
P6 tournament  0.108    0.125  0.118       1.0        1.0   1.0      1.0  1.0  
Hybrid         0.108    0.125  0.117       1.0        1.0   1.0      1.0  1.0  
Hybrid margin  0.102    0.124  0.118       1.0        1.0   1.0      1.0  1.0  

[4 rows x 25 columns]

Percentage of SKUs that keep the ensemble at each decision; the rest switch to an individual model.

In [34]:
kept = pd.Series({key: (choice == "Ens_top3").mean() * 100 for key, choice in picks.items()})
kept.index = pd.MultiIndex.from_tuples(kept.index, names=["strategy", "cutoff"])
kept.unstack("cutoff").loc[list(hybrid_policies)].round(0)

cutoff,2024-12-04,2025-01-01,2025-02-12,2025-03-12,2025-04-09
strategy,,,,,
Hybrid_portfolio,0.0,0.0,0.0,0.0,0.0
Hybrid_department,16.0,12.0,21.0,25.0,24.0
Hybrid_class,6.0,9.0,10.0,8.0,10.0
Hybrid_subclass,5.0,6.0,6.0,5.0,6.0
Hybrid_sku,2.0,2.0,2.0,2.0,2.0
Hybrid_margin_portfolio,100.0,100.0,100.0,100.0,100.0
Hybrid_margin_department,63.0,68.0,73.0,75.0,73.0
Hybrid_margin_class,44.0,51.0,56.0,53.0,56.0
Hybrid_margin_subclass,33.0,37.0,39.0,39.0,39.0


Consistency across the five decision dates (two development decisions, three later cutoffs). Changes are in WAPE points against the standalone ensemble; "beats P6 tournament" compares each policy with the plain tournament at the same level.

In [35]:
wape_c = wape_by_cutoff(hybrid_all, actuals, hybrid_decisions)
vs_ensemble = wape_c.loc[design.model] - wape_c.loc["Average_top3_ml"]
p6_same_level = design.level.map(tour[tour.pool == "P6_hist_drivers"].set_index("level").model)
vs_p6 = pd.DataFrame(wape_c.loc[design.model].values - wape_c.loc[p6_same_level].values, index=design.model, columns=wape_c.columns)
early, late = hybrid_decisions[:2], hybrid_decisions[2:]

pd.concat({
    "beats ensemble": by_family((vs_ensemble < 0).sum(axis=1)),
    "beats P6 tournament": by_family((vs_p6 < 0).sum(axis=1)),
    "change vs ensemble, development": by_family(vs_ensemble[early].mean(axis=1) * 100).round(2),
    "change vs ensemble, later cutoffs": by_family(vs_ensemble[late].mean(axis=1) * 100).round(2),
}, axis=1)

beats ensemble                                \
level              portfolio department class subclass sku   
family                                                       
P5 tournament              3          0     0        1   1   
P6 tournament              4          0     0        1   1   
Hybrid                     4          0     0        1   1   
Hybrid margin              0          0     0        1   2   

              beats P6 tournament                                \
level                   portfolio department class subclass sku   
family                                                            
P5 tournament                   0          1     2        1   1   
P6 tournament                   0          0     0        0   0   
Hybrid                          0          5     4        4   5   
Hybrid margin                   1          4     5        5   5   

              change vs ensemble, development                                  \
level                               portfolio department class subclass   sku   
family                                                                          
P5 tournament                           -0.66       2.50  1.53     0.04  0.10   
P6 tournament                           -0.69       1.02  1.35    -0.04 -0.20   
Hybrid                                  -0.69       0.90  1.28    -0.04 -0.20   
Hybrid margin                            0.00       0.62  0.79    -0.45 -0.75   

              change vs ensemble, later cutoffs                            \
level                                 portfolio department class subclass   
family                                                                      
P5 tournament                              0.08       1.58  1.93     2.08   
P6 tournament                             -0.08       1.60  1.88     1.95   
Hybrid                                    -0.08       1.47  1.88     1.92   
Hybrid margin                              0.00       1.52  1.58     1.43   

                     
level           sku  
family               
P5 tournament  3.10  
P6 tournament  2.99  
Hybrid         2.95  
Hybrid margin  2.62

- **Adding the ensemble to the pool changes little.** Hybrid is within 0.002 WAPE of the plain P6 tournament at every level. In the final window the tournament keeps the ensemble for 24% of SKUs at department level, 10% at class, 6% at subclass and 2% at SKU.
- **The 5% margin keeps the ensemble more often** (73%, 56%, 39% and 26% of SKUs). It is slightly better than the plain tournament at class (0.369 against 0.372), subclass (0.370 against 0.380) and SKU (0.388 against 0.391).
- **No selection policy beats the standalone ensemble (0.350) or the best individual model (0.349).**
- Five overlapping windows are weak evidence. The ensemble's leave-one-out evidence may make it look slightly worse than it is.

## F. Ensemble diversity

**Question.** The top-three average picks its members by WAPE, so it may average near-copies of one LightGBM. Would members with different errors make a better ensemble?

**What changes.** Which models are averaged. The pool is P7 plus the CatBoost section of notebook 03 (three CatBoost models and two L2 LightGBM models with lagged drivers), all without future drivers. Each model belongs to a family:

- **statistical**: Naive, HistoricAverage, SeasonalNaive
- **lgbm_l1**: LightGBM with the L1 objective (notebook 06)
- **lgbm_other**: LightGBM with other objectives, including the 03 and 03c models, horizon strategies and pooling
- **catboost**

**Why it might help.** Averaging cancels errors only when they point in different directions. Highly correlated members cancel little.

In [36]:
pd.set_option("display.max_colwidth", None)
boosting = load_forecasts(["03_catboost"])
candidates = pd.concat([candidates, boosting])
pool = nested["P7_behavior"] + sorted(boosting.model.unique())
assert set(pool) <= set(usable_models(candidates, future_drivers=False))

family = {m: "statistical" if m in stats else "catboost" if m.startswith("catboost") else "lgbm_l1" if m.startswith("lgbm_l1") else "lgbm_other" for m in pool}
ml_p7 = [m for m in nested["P7_behavior"] if m not in stats]

errors, y = residuals(candidates, actuals, pool, completed_runs(TEST_CUTOFF))
single = errors.abs().sum() / y.sum()
pd.DataFrame({"family": family, "dev_wape": single}).sort_values("dev_wape").groupby("family").head(3).round(3)

,family,dev_wape
lgbm_l1_hist_drivers,lgbm_l1,0.354
lgbm_l1_std_categorical,lgbm_l1,0.358
lgbm_l1_std,lgbm_l1,0.378
lgbm_direct,lgbm_other,0.398
SeasonalNaive,statistical,0.408
lgbm_hist_subclass_agg,lgbm_other,0.411
lgbm_hist_combined,lgbm_other,0.411
HistoricAverage,statistical,0.426
catboost_exog_categorical,catboost,0.434
catboost_hist_exog,catboost,0.438


How different are the errors? For the best two models of each family on the seven development runs: the correlation of forecast errors, and the share of product-weeks where two models err in opposite directions. Errors are in units, so high-volume products dominate the correlation, as they do WAPE.

In [37]:
shortlist = single.groupby(family).nsmallest(2).index.get_level_values(-1)
correlation = errors[shortlist].corr()
opposite = pd.DataFrame({a: {b: (np.sign(errors[a]) != np.sign(errors[b])).mean() for b in shortlist} for a in shortlist})
pd.concat({"error correlation": correlation, "opposite sign": opposite}, axis=1).round(2)

error correlation                     \
model                     catboost_exog_categorical catboost_hist_exog   
catboost_exog_categorical                      1.00               0.99   
catboost_hist_exog                             0.99               1.00   
lgbm_l1_hist_drivers                           0.84               0.83   
lgbm_l1_std_categorical                        0.85               0.84   
lgbm_direct                                    0.93               0.93   
lgbm_hist_subclass_agg                         0.93               0.92   
SeasonalNaive                                  0.62               0.61   
HistoricAverage                                0.89               0.89   

                                                                        \
model                     lgbm_l1_hist_drivers lgbm_l1_std_categorical   
catboost_exog_categorical                 0.84                    0.85   
catboost_hist_exog                        0.83                    0.84   
lgbm_l1_hist_drivers                      1.00                    0.99   
lgbm_l1_std_categorical                   0.99                    1.00   
lgbm_direct                               0.90                    0.91   
lgbm_hist_subclass_agg                    0.86                    0.87   
SeasonalNaive                             0.61                    0.61   
HistoricAverage                           0.89                    0.91   

                                                                            \
model                     lgbm_direct lgbm_hist_subclass_agg SeasonalNaive   
catboost_exog_categorical        0.93                   0.93          0.62   
catboost_hist_exog               0.93                   0.92          0.61   
lgbm_l1_hist_drivers             0.90                   0.86          0.61   
lgbm_l1_std_categorical          0.91                   0.87          0.61   
lgbm_direct                      1.00                   0.88          0.66   
lgbm_hist_subclass_agg           0.88                   1.00          0.64   
SeasonalNaive                    0.66                   0.64          1.00   
HistoricAverage                  0.94                   0.84          0.68   

                                                      opposite sign  \
model                     HistoricAverage catboost_exog_categorical   
catboost_exog_categorical            0.89                      0.00   
catboost_hist_exog                   0.89                      0.02   
lgbm_l1_hist_drivers                 0.89                      0.16   
lgbm_l1_std_categorical              0.91                      0.16   
lgbm_direct                          0.94                      0.15   
lgbm_hist_subclass_agg               0.84                      0.08   
SeasonalNaive                        0.68                      0.36   
HistoricAverage                      1.00                      0.21   

                                                                   \
model                     catboost_hist_exog lgbm_l1_hist_drivers   
catboost_exog_categorical               0.02                 0.16   
catboost_hist_exog                      0.00                 0.16   
lgbm_l1_hist_drivers                    0.16                 0.00   
lgbm_l1_std_categorical                 0.16                 0.07   
lgbm_direct                             0.15                 0.16   
lgbm_hist_subclass_agg                  0.09                 0.15   
SeasonalNaive                           0.37                 0.33   
HistoricAverage                         0.21                 0.18   

                                                               \
model                     lgbm_l1_std_categorical lgbm_direct   
catboost_exog_categorical                    0.16        0.15   
catboost_hist_exog                           0.16        0.15   
lgbm_l1_hist_drivers                         0.07        0.16   
lgbm_l1_std_categorical                     

The members of the P7 top-three average, `Average_top3_ml_behavior`, and their error correlations:

In [38]:
p7_top3 = top3(candidates, actuals, ml_p7, completed_runs(TEST_CUTOFF))
errors[p7_top3].corr().round(3)

model,lgbm_l1_hist_drivers,lgbm_l1_std_categorical,lgbm_l1_std
model,,,
lgbm_l1_hist_drivers,1.000,0.989,0.938
lgbm_l1_std_categorical,0.989,1.000,0.949
lgbm_l1_std,0.938,0.949,1.000


Four rules that build diversity in, each an equal-weight average (`ensemble_rules`):

- **C**: the best model and the best statistical model.
- **D**: the best model and the best model from another machine-learning family.
- **E**: the best model, then twice the competitive model (within 3 WAPE points of the best) least correlated with the members so far.
- **F**: the best model and the best model of each of the two next-best families.

A is the P7 top-three average and B the best individual model. Members chosen on all seven development runs:

In [39]:
fixed = ensemble_rules(candidates, actuals, family, completed_runs(TEST_CUTOFF))
fixed_members = {"A_P7_top3": p7_top3, "B_best_individual": [single.idxmin()]} | fixed
pd.DataFrame({
    "members": {name: ", ".join(members) for name, members in fixed_members.items()},
    "families": {name: ", ".join(family[m] for m in members) for name, members in fixed_members.items()},
    "mean_member_correlation": {name: mean_correlation(errors, members) for name, members in fixed_members.items()},
})

,members,families,mean_member_correlation
A_P7_top3,"lgbm_l1_hist_drivers, lgbm_l1_std_categorical, lgbm_l1_std","lgbm_l1, lgbm_l1, lgbm_l1",0.958496
B_best_individual,lgbm_l1_hist_drivers,lgbm_l1,NaN
Ensemble_C_best_plus_statistical,"lgbm_l1_hist_drivers, SeasonalNaive","lgbm_l1, statistical",0.605887
Ensemble_D_best_plus_other_ml,"lgbm_l1_hist_drivers, lgbm_direct","lgbm_l1, lgbm_other",0.900736
Ensemble_E_diverse,"lgbm_l1_hist_drivers, lgbm_l1_std, lgbm_l1_std_categorical","lgbm_l1, lgbm_l1, lgbm_l1",0.958496
Ensemble_F_family,"lgbm_l1_hist_drivers, lgbm_direct, SeasonalNaive","lgbm_l1, lgbm_other, statistical",0.721153


These ensembles scored on the same development runs that chose them. That flatters every rule, so it only shows whether a rule is competitive at all. WAPE by development origin, the mean change against the best individual model and its standard error over the seven origins (optimistic, because the origins overlap by ten weeks).

In [40]:
cutoff_level = errors.index.get_level_values("cutoff")
by_origin = pd.DataFrame({name: errors[members].mean(axis=1).abs().groupby(cutoff_level).sum() / y.groupby(cutoff_level).sum()
                          for name, members in fixed_members.items()}).T
change = (by_origin - by_origin.loc["B_best_individual"]) * 100
by_origin.round(3).assign(
    dev_wape=pd.Series({name: errors[m].mean(axis=1).abs().sum() / y.sum() for name, m in fixed_members.items()}).round(3),
    change_points=change.mean(axis=1).round(2),
    standard_error=(change.std(axis=1) / np.sqrt(change.shape[1])).round(2),
    beats_best=(change < 0).sum(axis=1),
)

cutoff,2024-07-17 00:00:00,2024-08-14 00:00:00,2024-09-11 00:00:00,2024-10-09 00:00:00,2024-11-06 00:00:00,2024-12-04 00:00:00,2025-01-01 00:00:00,dev_wape,change_points,standard_error,beats_best
A_P7_top3,0.345,0.339,0.377,0.361,0.350,0.387,0.346,0.358,0.39,0.13,0
B_best_individual,0.339,0.337,0.374,0.360,0.348,0.376,0.343,0.354,0.00,0.00,0
Ensemble_C_best_plus_statistical,0.368,0.363,0.367,0.348,0.337,0.341,0.341,0.352,-0.17,0.84,5
Ensemble_D_best_plus_other_ml,0.366,0.351,0.383,0.368,0.360,0.381,0.359,0.367,1.32,0.27,0
Ensemble_E_diverse,0.345,0.339,0.377,0.361,0.350,0.387,0.346,0.358,0.39,0.13,0
Ensemble_F_family,0.372,0.360,0.371,0.354,0.344,0.352,0.347,0.357,0.32,0.72,4


Time-valid version: the same rules applied at each of the six decision dates with only the runs completed by then, scored on the final 14 weeks (exploratory) next to A and B.

In [41]:
rolling = {d: ensemble_rules(candidates, actuals, family, completed_runs(d)) for d in all_decisions}
ensemble_runs = pd.concat([average(name, candidates, members, [d]) for d, rules in rolling.items() for name, members in rules.items()])
benchmarks = pd.concat([average("Average_top3_ml_behavior", candidates, top3(candidates, actuals, ml_p7, completed_runs(d)), [d]) for d in all_decisions]
                       + [rows for (p, d), (rows, name) in best.items() if p == "P7_behavior"])
diversity_compare = pd.concat([benchmarks, ensemble_runs])

diversity_board = leaderboard(diversity_compare, actuals, scope="test").set_index("model")
diversity_board[["wape", "bias", "mean_rmsse", "median_rmsse", "apc", "coverage"]].round(3)

,wape,bias,mean_rmsse,median_rmsse,apc,coverage
model,,,,,,
Best_P7_behavior,0.349,-0.028,1.255,0.671,0.116,1.0
Average_top3_ml_behavior,0.350,-0.017,1.251,0.661,0.111,1.0
Ensemble_E_diverse,0.350,-0.017,1.251,0.661,0.111,1.0
Ensemble_F_family,0.354,-0.033,1.281,0.699,0.077,1.0
Ensemble_D_best_plus_other_ml,0.356,-0.031,1.262,0.671,0.117,1.0
Ensemble_C_best_plus_statistical,0.358,-0.033,1.307,0.735,0.058,1.0


Change in WAPE points against the best individual model at each decision date.

In [42]:
diversity_wape = wape_by_cutoff(diversity_compare, actuals, all_decisions)
change = (diversity_wape - diversity_wape.loc["Best_P7_behavior"]) * 100
change.drop(index="Best_P7_behavior").round(2).assign(beats_best=(change.drop(index="Best_P7_behavior") < 0).sum(axis=1))

cutoff,2024-11-06 00:00:00,2024-12-04 00:00:00,2025-01-01 00:00:00,2025-02-12 00:00:00,2025-03-12 00:00:00,2025-04-09 00:00:00,beats_best
model,,,,,,,
Average_top3_ml_behavior,0.20,1.11,0.27,0.26,-0.16,0.13,1
Ensemble_C_best_plus_statistical,2.73,1.14,2.23,1.64,0.80,0.92,0
Ensemble_D_best_plus_other_ml,1.62,0.54,1.64,1.18,0.17,0.65,0
Ensemble_E_diverse,1.25,1.11,0.27,0.26,-0.16,0.13,1
Ensemble_F_family,2.35,1.21,2.78,1.46,0.28,0.52,0


Members chosen at each decision date. Rules whose members change from date to date are less settled than the fixed table suggests.

In [43]:
pd.DataFrame({d.date(): {name: ", ".join(members) for name, members in rules.items()} for d, rules in sorted(rolling.items())}).T

,Ensemble_C_best_plus_statistical,Ensemble_D_best_plus_other_ml,Ensemble_E_diverse,Ensemble_F_family
2024-11-06,"lgbm_l1_hist_drivers, Naive","lgbm_l1_hist_drivers, lgbm_tweedie_raw","lgbm_l1_hist_drivers, lgbm_l1_raw, lgbm_l1_std","lgbm_l1_hist_drivers, lgbm_tweedie_raw, Naive"
2024-12-04,"lgbm_l1_hist_drivers, HistoricAverage","lgbm_l1_hist_drivers, lgbm_direct","lgbm_l1_hist_drivers, lgbm_l1_std, lgbm_l1_std_categorical","lgbm_l1_hist_drivers, lgbm_direct, HistoricAverage"
2025-01-01,"lgbm_l1_hist_drivers, HistoricAverage","lgbm_l1_hist_drivers, lgbm_direct","lgbm_l1_hist_drivers, lgbm_l1_std, lgbm_l1_std_categorical","lgbm_l1_hist_drivers, lgbm_direct, HistoricAverage"
2025-02-12,"lgbm_l1_hist_drivers, SeasonalNaive","lgbm_l1_hist_drivers, lgbm_direct","lgbm_l1_hist_drivers, lgbm_l1_std, lgbm_l1_std_categorical","lgbm_l1_hist_drivers, lgbm_direct, SeasonalNaive"
2025-03-12,"lgbm_l1_hist_drivers, SeasonalNaive","lgbm_l1_hist_drivers, lgbm_direct","lgbm_l1_hist_drivers, lgbm_l1_std, lgbm_l1_std_categorical","lgbm_l1_hist_drivers, lgbm_direct, SeasonalNaive"
2025-04-09,"lgbm_l1_hist_drivers, SeasonalNaive","lgbm_l1_hist_drivers, lgbm_direct","lgbm_l1_hist_drivers, lgbm_l1_std, lgbm_l1_std_categorical","lgbm_l1_hist_drivers, lgbm_direct, SeasonalNaive"


- **The top three are near-duplicates.** `lgbm_l1_hist_drivers`, `lgbm_l1_std_categorical` and `lgbm_l1_std` have error correlations of 0.94 to 0.99, so their average adds little to the best of them. Rule E picks the same three. The most different competitive errors come from seasonal naive (mean correlation 0.61 in rule C).
- **Even on the runs that chose them, no ensemble beats the best model on average.** C (best plus a statistical model) is level (-0.17 points, standard error 0.84), with large swings. A and E are 0.4 points worse and D is 1.3 points worse.
- **Time-valid rolling selection is worse still.** At the six decision dates, C, D and F never beat the best individual model, and A and E beat it once. Final window: best individual 0.349, A and E 0.350, F 0.354, D 0.356, C 0.358.
- **Diversity buys stability, not accuracy.** With seasonal naive in the mix the forecasts revise much less: APC 0.058 for C and 0.077 for F, against 0.116 for the best individual model.

On this evidence the best model on its own is the supported choice. If stability matters, F is the ensemble to consider; it costs about half a point of WAPE.

Save the strategies at the final and stability cutoffs for notebook 08. This needs every optional section.

In [44]:
saved = pd.concat([strategies, hier_runs[hier_runs.cutoff.isin(targets)], tour_runs[tour_runs.cutoff.isin(targets)],
                   pool_averages[pool_averages.cutoff.isin(targets)], hybrid_runs[hybrid_runs.cutoff.isin(targets)],
                   benchmarks[(benchmarks.model == "Average_top3_ml_behavior") & benchmarks.cutoff.isin(targets)], hindsight])
save_forecasts(saved, "07_model_selection")